Put all the Fish PFAS data together   
First, collate the data from different sources

In [2]:
import pandas as pd
path_nrsp = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\"
path_caea = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\caea\\"
path_egle = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\egle\\"
path_epa = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\epa\\"
path_norman = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\norman\\"
path_paper = "D:\\wyy\\pyrunning\\spdb_sae\\new_resource_sp\\paper\\"
path_raw = "C:\\Users\\laowu\\OneDrive\\file\\"

In [ ]:
df_paper = pd.read_csv(path_paper + 'paper.csv')
df_paper['source'] = 'paper'
df_norman = pd.read_csv(path_norman + 'norman.csv')
df_norman['source'] = 'norman'
df_caea = pd.read_csv(path_caea + 'caea.csv')
df_caea['source'] = 'caea'
df_epa = pd.read_csv(path_epa + 'epa.csv')
df_epa['source'] = 'epa'
df_egle = pd.read_csv(path_egle + 'egle.csv')
df_egle['source'] = 'egle'

df_sp = pd.concat([df_paper, df_norman, df_caea, df_epa, df_egle],axis=0)
df_sp['n'] = df_sp['n'].fillna(1)

df_sp = df_sp[df_sp['lon'].notna()]

df_sp.to_csv(path_nrsp + 'lr_raw.csv', index=False)

In [ ]:
import pandas as pd

df_lr_raw = pd.read_csv(path_nrsp + 'lr_raw.csv')


def calculate_limit_percentage_and_type(df, limits):
    type_counts = df['type'].value_counts()
    type_percentage = df['type'].value_counts(normalize=True) * 100
    type_info = pd.DataFrame({
        'Count': type_counts,
        'Percentage': type_percentage
    })

    result = [] 
    for limit in limits:
        count = df[df['limit_value'] > limit].shape[0]
        percentage = (count / df.shape[0]) * 100
        result.append({
            'Condition': f'limit_value > {limit}',
            'Count': count,
            'Percentage': percentage
        })
    limit_info = pd.DataFrame(result)

    return type_info, limit_info

type_info, limit_info = calculate_limit_percentage_and_type(df_lr_raw, [0.01, 0.1, 0.5, 1])
print("Type Information:\n", type_info)
print("\nLimit Value Information:\n", limit_info)


Type Information:
    Count  Percentage
0  38415   52.749018
1  34411   47.250982

Limit Value Information:
             Condition  Count  Percentage
0  limit_value > 0.01  37189   51.065554
1   limit_value > 0.1  21844   29.994782
2   limit_value > 0.5   7518   10.323236
3     limit_value > 1   4744    6.514157


#### caea

In [ ]:



po_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='po_pfas')
po_dict = dict(zip(po_df['po_name'], po_df['poid']))

sp2_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='sp_pfas')
sp2_dict = dict(zip(sp2_df['spid'], sp2_df['canonicalName']))

rp_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='rp')
sp_df = rp_df[rp_df['type'] == 'sp']
sp_dict = dict(zip(sp_df['NAME'], sp_df['ID']))

file2_df = pd.read_csv(path_caea + "caea_raw.csv")

file2_df['ID'] = file2_df['Species'].map(sp_dict)

file2_df['poid'] = file2_df['posname'].map(po_dict)
file2_df = file2_df.rename(columns={'ID': 'spid'})
file2_df['canonicalName'] = file2_df['spid'].map(sp2_dict)
file2_df['n'] = 1
file2_df['habit'] = 'R'
file2_df['paid'] = 1560

file2_df.to_csv(path_caea + "caea_re.csv", index=False)

df_raw = pd.read_csv(path_caea + "caea_re.csv")
df_raw['type'] = 0  # 默认为0
df_raw['limit_value'] = None  # 默认为空白

df_raw.loc[df_raw['remark'] == '<MDL', 'type'] = 1
df_raw.loc[df_raw['remark'] == '<MDL', 'limit_value'] = df_raw['value'] * 2

df_raw.loc[df_raw['remark'].isna(), 'type'] = 0
df_raw.loc[df_raw['remark'].isna(), 'limit_value'] = None

df_raw = df_raw[['poid','spid','paid','year','lon','lat','habit','value','n','organ', 'limit_value', 'type']]
df_raw.to_csv(path_caea + "caea.csv", index=False)



In [ ]:


po_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='po_pfas')
po_dict = dict(zip(po_df['po_name'], po_df['poid']))


sp2_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='sp_pfas')
sp2_dict = dict(zip(sp2_df['spid'], sp2_df['canonicalName']))

rp_df = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='rp')
sp_df = rp_df[rp_df['type'] == 'sp']
sp_dict = dict(zip(sp_df['NAME'], sp_df['ID']))

file2_df = pd.read_csv(path_egle + 'egle_raw.csv')

file2_df['ID'] = file2_df['Species'].map(sp_dict)
file2_df['poid'] = file2_df['posname'].map(po_dict)



file2_df = file2_df.rename(columns={'ID': 'spid'})

file2_df['canonicalName'] = file2_df['spid'].map(sp2_dict)
file2_df['n'] = 1
file2_df['paid'] = 1561


file2_df = file2_df[['poid','spid','paid','year','lon','lat','habit','value','n','organ', 'limit_value', 'type']]
file2_df.to_csv(path_egle + "egle.csv", index=False)

#### epa

In [ ]:
import pandas as pd

df_pfas = pd.read_excel(path_epa + 'raw\\merge_excel_deal.xlsx', sheet_name='pfas')
df_sample = pd.read_excel(path_epa + 'raw\\merge_excel_deal.xlsx', sheet_name='sample')
df_rp = pd.read_excel(path_raw + 'inf.xlsx', sheet_name='rp')
print(df_sample.columns)

sample_dict_time = df_sample.set_index('Sample_ID')['year'].to_dict()
sample_dict_n = df_sample.set_index('Sample_ID')['n'].to_dict()
sample_dict_Scientific_Name = df_sample.set_index('Sample_ID')['Scientific_Name'].to_dict()


site_dict_crd_type = df_sample.set_index('Site_ID')['crd_type'].to_dict()
site_dict_lon = df_sample.set_index('Site_ID')['lon'].to_dict()
site_dict_lat = df_sample.set_index('Site_ID')['lat'].to_dict()


df_pfas['year'] = df_pfas['Sample_ID'].map(sample_dict_time)
df_pfas['n'] = df_pfas['Sample_ID'].map(sample_dict_n)
df_pfas['Scientific_Name'] = df_pfas['Sample_ID'].map(sample_dict_Scientific_Name)


df_pfas['crd_type'] = df_pfas['Site_ID'].map(site_dict_crd_type)
df_pfas['lon'] = df_pfas['Site_ID'].map(site_dict_lon)
df_pfas['lat'] = df_pfas['Site_ID'].map(site_dict_lat)



df_pfas['poid'] = None
df_pfas['spid'] = None


df_pfas['po_name'] = df_pfas['po_name'].str.lower()
df_pfas['Scientific_Name'] = df_pfas['Scientific_Name'].str.lower()
df_rp['NAME'] = df_rp['NAME'].str.lower()

for i, row in df_pfas.iterrows():
    temp = df_rp[df_rp['NAME'] == row['po_name']]['ID']
    if temp.nunique() == 1:
        df_pfas.at[i, 'poid'] = temp.values[0]
    temp = df_rp[df_rp['NAME'] == row['Scientific_Name']]['ID']
    if temp.nunique() == 1:
        df_pfas.at[i, 'spid'] = temp.values[0]
    if pd.isnull(row['deal']):
        continue
    elif row['deal'] == 'MDL':
        df_pfas.at[i, 'Value'] = row['MDL'] / 2
    elif row['deal'] == 'QL':
        df_pfas.at[i, 'Value'] = row['QL'] / 2
    elif row['deal'] == 'BET':
        df_pfas.at[i, 'Value'] = (row['MDL'] + row['QL']) / 2

state_dict = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California', 
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia', 
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 
    'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 
    'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 
    'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee', 
    'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 
    'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

df_pfas['province_e'] = df_pfas['State'].map(state_dict)


df_pfas.to_csv(path_epa + "epa_raw.csv", index=False, encoding="utf-8-sig")


Index(['EPA_Region', 'State', 'Location', 'crd_type', 'Site_ID', 'lat', 'lon',
       'Collection_Date', 'Sample_ID', 'Family', 'Scientific_Name',
       'Common_Name', 'year', 'Total_Length', 'n'],
      dtype='object')


In [ ]:
df_pfas = pd.read_csv(path_epa + "epa_raw.csv")

filtered_df = df_pfas[~df_pfas['Qualifier_Flag'].str.contains('[HBNK]', na=False)]
filtered_df = filtered_df[(filtered_df['poid'].notna())&(filtered_df['spid'].notna())]

filtered_df['type'] = 0  # 默认为0
filtered_df['paid'] = 1553
filtered_df['organ'] = 'muscle'
filtered_df.loc[filtered_df['Qualifier_Flag'] == 'U', 'type'] = 1

filtered_df = filtered_df.rename(columns={'MDL': 'limit_value', 'crd_type': 'habit', 'Value':'value'})

filtered_df = filtered_df[['poid','spid','paid','year','lon','lat','habit','value','n','organ', 'limit_value', 'type']]
filtered_df.to_csv(path_epa + "epa.csv", index=False)

#### norman

In [ ]:
import pandas as pd

df_n = pd.read_csv(path_norman + 'raw\\biota_norman.csv')
df_n = df_n[['lon','lat','habit','value','n', 'organ','year','limit_value', 'type','CAS','NAME']]


df_n[["CAS","NAME"]] = df_n[["CAS","NAME"]].astype(str)
df_n["NAME"] = df_n["NAME"].str.strip()
df_n["CAS"] = df_n["CAS"].str.strip()
df_po = pd.read_excel(path_raw + "inf.xlsx", sheet_name='po')
df_po[["CAS","id"]] = df_po[["CAS","id"]].astype(str)

df_po = df_po[df_po["CAS"].notna()]
print(df_po[["CAS","id"]].head())
dict_poid = df_po.set_index('CAS')['id'].to_dict()

df_sp = pd.read_excel(path_raw + "inf.xlsx", sheet_name='rp')
df_sp[["NAME","ID"]] = df_sp[["NAME","ID"]].astype(str)
dict_spid = df_sp.set_index('NAME')['ID'].to_dict()

df_n["poid"] = df_n["CAS"].map(dict_poid)
df_n["spid"] = df_n["NAME"].map(dict_spid)
df_n['paid'] = 1554


df_n = df_n[(df_n['spid'].notna())&(df_n['spid'].notna())]

df_n['organ'] = df_n['organ'].replace({'Whole body': 'whole', 'Muscle': 'muscle', 'Eggs':'egg', 'Liver':'liver'})

df_n = df_n[['poid','spid','paid', 'lon','lat','habit','value','n', 'organ','year','limit_value', 'type']]
df_n.to_csv(path_norman + 'norman.csv', index=False)



         CAS  id
0  7439-97-6  21
1  7439-92-1  22
2  7440-43-9  23
3  7440-38-2  24
4  7440-50-8  25


#### PAPER

In [ ]:
import pandas as pd
import re

df_paper = pd.read_csv(path_paper + 'raw\\last_lr.csv')
df_paper = df_paper[['id','poid','spid','paid','time','organ','crd_type','lon','lat','avg','n','dw','unit','limit_value','type']]

df_paper['time'] = df_paper['time'].str.replace(" ", "")

df_paper['str_year'] = df_paper['time'].str.len()

df_paper = df_paper[df_paper['str_year'].isin([4, 6, 9, 13])]


def process_time(row):
    if row['str_year'] == 4:
        return row['time']

    elif row['str_year'] == 6:
        return row['time'][:4]  # 只保留前4个字符
    elif row['str_year'] in [9, 13]:
        split_values = re.split(r'[\/\-\–]', row['time'])

        if len(split_values) == 2 and split_values[0] == split_values[1]:  # 若前后列相等
            if row['str_year'] == 9:
                return split_values[0]  # 9位时，取前列值
            elif row['str_year'] == 13:
                return split_values[0][:4]  # 13位时，取前列值的前4个字符
        return None  # 若前后列不相等，则不保留数据
    return row['time']


df_paper['year'] = df_paper.apply(process_time, axis=1)
df_paper.to_csv(path_paper + 'paper_raw.csv', index=False)


C:\Users\laowu\AppData\Local\Temp\ipykernel_24996\1990026250.py:5: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_paper = pd.read_csv(path_paper + 'raw\\last_lr.csv')


In [ ]:
import pandas as pd
df_paper_raw = pd.read_csv(path_paper + 'paper_raw.csv')
df_organ = pd.read_excel(path_paper + 'time_organ.xlsx',sheet_name='organ')

df_paper_raw = df_paper_raw.merge(df_organ[['organ', 're_organ']], on='organ', how='left')

df_paper_raw['organ'] = df_paper_raw['re_organ']

df_paper_raw.drop(columns=['re_organ'], inplace=True)
df_paper_raw = df_paper_raw[df_paper_raw['year'].notna()]
df_paper_raw['dw'] = df_paper_raw['dw'].str.lower()
df_paper_raw = df_paper_raw[(df_paper_raw['dw']=='ww')&(df_paper_raw['unit']=='ng/g')]

df_paper_raw = df_paper_raw.rename(columns={'avg':'value', 'crd_type':'habit'})
df_paper_raw = df_paper_raw[['poid','spid','paid','year','organ', 'habit','lon','lat','value','n','limit_value','type']]
list_del = [1306, 1663]

df_paper_raw = df_paper_raw[~df_paper_raw['paid'].isin(list_del)]

df_paper_raw.to_csv(path_paper + 'paper.csv', index=False)
